# 06 — Model 2: Random Forest


> **Notebook 6 of 11** — part of the *Heart Disease Detection using Explainable AI* project.
> Run the notebooks **in order**, from 01 to 11.

---

## 🎯 How a Random Forest works, explained simply

A **decision tree** asks a chain of yes/no questions:

```
Is systolic BP above 140?
   └─ yes → Is age above 55?
              └─ yes → probably HEART DISEASE
```

One tree alone is easily fooled by noise. So a **Random Forest grows 200 different trees**, each
shown a slightly different random slice of the data and a random subset of features, then lets
them **vote**. The crowd is far wiser than any individual — this is called *ensemble learning*.

| Strength | Weakness |
|---|---|
| Captures curves and feature combinations | Slower, and harder to interpret directly |
| Very resistant to overfitting | Bigger file on disk |

## ⚙️ The settings we tune

* **`max_depth`** — how many questions deep each tree may go. Too deep and it memorises the
  training set.
* **`min_samples_leaf`** — the minimum number of patients allowed at the end of a branch. Larger
  values stop the tree inventing rules from two or three lucky patients.

> ⏱️ This is the slowest notebook — expect roughly **2–5 minutes**.

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve, classification_report)

warnings.filterwarnings("ignore"); sns.set_style("whitegrid")
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
DATA, MODELS = "../data", "../models"

# --- load what notebook 04 prepared ---
prep = np.load(f"{DATA}/prepared.npz", allow_pickle=True)
FEATURES = list(prep["features"])
X_train, X_test = prep["X_train"], prep["X_test"]
y_train, y_test = prep["y_train"], prep["y_test"]
scaler = joblib.load(f"{MODELS}/scaler.pkl")

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)
cv_plan = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"Training patients : {len(X_train):,}")
print(f"Test patients     : {len(X_test):,}")
print(f"Features          : {FEATURES}")

from sklearn.ensemble import RandomForestClassifier

In [ ]:
def save_results(model_name, model, best_params, cv_scores, seconds):
    """Grade the model on the SEALED test set and append the scores to models/metrics.json."""
    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    keep = max(1, len(fpr) // 200)

    entry = {
        "accuracy":  float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred)),
        "recall":    float(recall_score(y_test, y_pred)),
        "f1":        float(f1_score(y_test, y_pred)),
        "roc_auc":   float(roc_auc_score(y_test, y_proba)),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "fpr": fpr[::keep].tolist(), "tpr": tpr[::keep].tolist(),
        "cv_scores": [float(s) for s in cv_scores],
        "cv_mean": float(np.mean(cv_scores)), "cv_std": float(np.std(cv_scores)),
        "best_params": {k: str(v) for k, v in best_params.items()},
        "train_seconds": round(seconds, 1),
    }

    path = f"{MODELS}/metrics.json"
    all_metrics = json.load(open(path)) if os.path.exists(path) else {}
    all_metrics[model_name] = entry
    json.dump(all_metrics, open(path, "w"), indent=2)

    print(f"\n{'=' * 58}\n  {model_name}\n{'=' * 58}")
    print(f"  Accuracy   : {entry['accuracy']:.4f}   (out of 100 patients, "
          f"{entry['accuracy']*100:.0f} labelled correctly)")
    print(f"  Precision  : {entry['precision']:.4f}   (when we say 'disease', we are right this often)")
    print(f"  Recall     : {entry['recall']:.4f}   (of all truly sick, we caught this many)")
    print(f"  F1 Score   : {entry['f1']:.4f}   (balance of precision and recall)")
    print(f"  ROC-AUC    : {entry['roc_auc']:.4f}   (0.5 = coin toss, 1.0 = perfect)")
    print(f"  CV accuracy: {entry['cv_mean']:.4f} +/- {entry['cv_std']:.4f}")
    return y_proba

print("Helper ready.")

## 1. Hyperparameter tuning with GridSearchCV

In [ ]:
print("Tuning Random Forest... this is the slow one, please wait (2-5 minutes)")
start = time.time()

grid = GridSearchCV(
    estimator=RandomForestClassifier(n_estimators=200, max_features="sqrt",
                                     random_state=RANDOM_STATE, n_jobs=-1),
    param_grid={"max_depth": [8, 10], "min_samples_leaf": [10, 30]},
    cv=3,                 # 3 folds instead of 5 here, purely to save time
    scoring="roc_auc", n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print(f"\nBest settings   : {grid.best_params_}")
print(f"Best CV ROC-AUC : {grid.best_score_:.4f}")
print(f"Took {time.time()-start:.0f} seconds")

res = pd.DataFrame(grid.cv_results_)
param_cols = [c for c in res.columns if c.startswith("param_")]
res[param_cols + ["mean_test_score"]].sort_values("mean_test_score",
                                                  ascending=False).round(4)

Look at the pattern in that table: deeper trees (`max_depth=10`) and smaller leaves score better,
but only slightly. The forest is nowhere near overfitting, which tells us the dataset has a clear,
simple signal rather than a complicated hidden one.

## 2. Cross-validation

In [ ]:
base_model = RandomForestClassifier(n_estimators=200, max_features="sqrt",
                                    random_state=RANDOM_STATE, n_jobs=-1,
                                    **grid.best_params_)
cv_scores = cross_val_score(base_model, X_train_scaled, y_train,
                            cv=cv_plan, scoring="accuracy", n_jobs=-1)

print("Accuracy on each of the 5 folds:", np.round(cv_scores, 4))
print(f"Average   : {cv_scores.mean():.4f}")
print(f"Variation : +/- {cv_scores.std():.4f}")

## Why `CalibratedClassifierCV`?

A model's raw output is a **score**, not necessarily an honest probability. A model might output
0.80 for a group of patients of whom only 65% are really sick — the *ranking* is right but the
*number* is exaggerated, and different algorithms exaggerate differently.

That matters for us, because the website shows all three percentages **side by side**. If one says
45% and another says 78%, the user cannot tell whom to believe.

**Isotonic calibration** learns a correction curve on held-out folds and re-maps every score so
that among patients given 70%, roughly 70% really are sick. Notebook 08 measures how well this
worked.


Random Forests are the classic example of a **miscalibrated** model. Because they average votes
from 200 trees, their probabilities cluster towards the middle and rarely reach 0 or 1. Calibration
matters most here.

In [ ]:
final_model = CalibratedClassifierCV(base_model, cv=5, method="isotonic")
final_model.fit(X_train_scaled, y_train)

proba = save_results("Random Forest", final_model,
                     grid.best_params_, cv_scores, time.time() - start)

In [ ]:
print(classification_report(y_test, final_model.predict(X_test_scaled),
                            target_names=["Healthy", "Heart disease"], digits=4))

cm = confusion_matrix(y_test, final_model.predict(X_test_scaled))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Greens", cbar=False, ax=ax,
            xticklabels=["Predicted\nHealthy", "Predicted\nDisease"],
            yticklabels=["Actually\nHealthy", "Actually\nDisease"])
ax.set_title("Random Forest — confusion matrix")
plt.tight_layout(); plt.show()

## 3. Built-in feature importance — and its limitation

A Random Forest can report how often each feature was used to split a branch. It is useful, but
**limited**, and knowing the limitation is worth marks.

In [ ]:
plain = base_model.fit(X_train_scaled, y_train)

imp = pd.DataFrame({"Feature": FEATURES,
                    "Importance": plain.feature_importances_}).sort_values("Importance")

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(imp.Feature, imp.Importance, color="#10B981")
ax.set_title("Random Forest built-in feature importance")
ax.set_xlabel("Importance")
plt.tight_layout(); plt.show()

print(imp.sort_values("Importance", ascending=False).to_string(index=False))

### ⚠️ What this chart CANNOT tell you

1. **No direction.** It says `ap_hi` matters, but not whether *high* BP raises or lowers risk.
2. **No individual patients.** It describes the forest overall, never one specific person.
3. **It is biased** towards features with many distinct values — continuous features like `bmi`
   look more important than binary ones like `smoke` partly for mathematical reasons rather than
   medical ones.

**This is exactly the gap SHAP and LIME fill** in notebooks 09 and 10. Being able to explain *why*
you did not stop at this chart is a strong viva answer.

In [ ]:
joblib.dump(final_model, f"{MODELS}/random_forest.pkl", compress=3)
np.save(f"{MODELS}/proba__random_forest.npy", proba)
print(f"Saved {MODELS}/random_forest.pkl  (compressed)")

---
## ✅ Summary

* 200 trees, tuned with GridSearchCV, validated with 5-fold cross-validation.
* Calibrated — which matters most for a forest.
* Usually the **best of the three** models, with ROC-AUC around **0.80**.
* Its built-in importance is useful but cannot give direction or explain one patient.

### ▶️ Next: `07_SVM.ipynb`